# Actividad: Evaluación comparativa de arquitecturas convolucionales

Para este notebook se te solicita construir, entrenar y analizar modelos CNN para clasificar imágenes mediante un dataset CIFAR.

**Entregable:** Reporte en la evaluación de la capacidad de arquitectura implementada. Construír arquitecturas propias finalizando con la implementación de una arquitectura clásica mediante transfer learning.


## Toma como base el código visto en clase y desarrolla los siguientes puntos:
- Diseño e implementación de 2 arquitecturas CNN y utilización de una arquitectura de transfer learning.

- Buen uso de data augmentation y regularización.

- Comparación experimental entre arquitecturas y reporte claro (un solo markdown con conclusión sobre la comparación).





In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
# Selecciona la arquitectura que desees; puedes buscar otras.
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2


## Definiciones de modelos

In [6]:
# Los modelos reciben imágenes RGB CIFAR en rango [0, 255] y producen 10 clases.
INPUT_SHAPE = (32, 32, 3)
NUM_CLASSES = 10

def crear_aumento():
    return keras.Sequential(
        [
            keras.layers.RandomFlip("horizontal"),
            keras.layers.RandomRotation(0.08),
            keras.layers.RandomZoom(0.1),
        ],
        name="aumento_datos",
    )

def crear_cnn_basica(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = crear_aumento()(inputs)
    x = keras.layers.Rescaling(1.0 / 255)(x)

    for filtros in (32, 64, 128):
        x = keras.layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
        x = keras.layers.Conv2D(filtros, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.Activation("relu")(x)
        x = keras.layers.MaxPooling2D()(x)
        x = keras.layers.Dropout(0.2)(x)

    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="cnn_basica")

def bloque_residual(x, filtros, stride=1):
    atajo = x
    y = keras.layers.Conv2D(
        filtros, 3, strides=stride, padding="same", use_bias=False
    )(x)
    y = keras.layers.BatchNormalization()(y)
    y = keras.layers.Activation("relu")(y)
    y = keras.layers.Conv2D(filtros, 3, padding="same", use_bias=False)(y)
    y = keras.layers.BatchNormalization()(y)

    if stride != 1 or x.shape[-1] != filtros:
        atajo = keras.layers.Conv2D(
            filtros, 1, strides=stride, padding="same", use_bias=False
        )(atajo)
        atajo = keras.layers.BatchNormalization()(atajo)

    y = keras.layers.Add()([y, atajo])
    return keras.layers.Activation("relu")(y)

def crear_cnn_residual(input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=input_shape)
    x = crear_aumento()(inputs)
    x = keras.layers.Rescaling(1.0 / 255)(x)
    x = keras.layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Activation("relu")(x)

    x = bloque_residual(x, 32)
    x = bloque_residual(x, 64, stride=2)
    x = bloque_residual(x, 128, stride=2)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dense(128, activation="relu")(x)
    x = keras.layers.Dropout(0.4)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="cnn_residual")

def crear_modelo_transfer_learning(
    input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, weights="imagenet"
):
    inputs = keras.Input(shape=input_shape)
    x = crear_aumento()(inputs)
    x = keras.layers.Resizing(96, 96)(x)
    x = keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)

    base = MobileNetV2(
        input_shape=(96, 96, 3), include_top=False, weights=weights
    )
    base.trainable = False
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="mobilenetv2_transfer")

modelo_cnn_basica = crear_cnn_basica()
modelo_cnn_residual = crear_cnn_residual()
modelo_transfer = crear_modelo_transfer_learning()

## Entrenamiento de modelos.

In [7]:
from time import perf_counter

SEED = 42
EPOCHS = 10
BATCH_SIZE = 128

keras.utils.set_random_seed(SEED)
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")

modelos = {
    "CNN básica": modelo_cnn_basica,
    "CNN residual": modelo_cnn_residual,
    "MobileNetV2 (transferencia)": modelo_transfer,
}
historiales = {}
resultados = {}

for nombre, modelo in modelos.items():
    print(f"\nEntrenando {nombre}...")
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=3, restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5
        ),
    ]

    inicio = perf_counter()
    history = modelo.fit(
        x_train,
        y_train,
        validation_split=0.1,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    minutos = (perf_counter() - inicio) / 60
    historiales[nombre] = history.history

    mejor_epoca = int(np.argmax(history.history["val_accuracy"]))
    test_loss, test_accuracy = modelo.evaluate(
        x_test, y_test, batch_size=BATCH_SIZE, verbose=0
    )
    resultados[nombre] = {
        "train_accuracy": float(history.history["accuracy"][mejor_epoca]),
        "val_accuracy": float(history.history["val_accuracy"][mejor_epoca]),
        "test_accuracy": float(test_accuracy),
        "test_loss": float(test_loss),
        "duracion_min": minutos,
        "epoca_mejor_validacion": mejor_epoca + 1,
    }
    print(
        f"{nombre}: validación={resultados[nombre]['val_accuracy']:.3f}, "
        f"prueba={test_accuracy:.3f}, tiempo={minutos:.1f} min"
    )

170498071/170498071 [==============================] - 890s 5us/step

Entrenando CNN básica...
Epoch 1/10
352/352 [==============================] - 18s 48ms/step - loss: 1.6397 - accuracy: 0.3989 - val_loss: 1.9346 - val_accuracy: 0.2952 - lr: 0.0010
Epoch 2/10
352/352 [==============================] - 16s 47ms/step - loss: 1.3229 - accuracy: 0.5241 - val_loss: 1.5147 - val_accuracy: 0.5018 - lr: 0.0010
Epoch 3/10
352/352 [==============================] - 16s 47ms/step - loss: 1.1851 - accuracy: 0.5801 - val_loss: 1.1652 - val_accuracy: 0.5870 - lr: 0.0010
Epoch 4/10
352/352 [==============================] - 16s 47ms/step - loss: 1.0953 - accuracy: 0.6179 - val_loss: 1.2142 - val_accuracy: 0.5932 - lr: 0.0010
Epoch 5/10
352/352 [==============================] - 17s 47ms/step - loss: 1.0193 - accuracy: 0.6430 - val_loss: 1.2352 - val_accuracy: 0.5900 - lr: 0.0010
Epoch 6/10
352/352 [==============================] - 16s 46ms/step - loss: 0.9211 - accuracy: 0.6804 - val_loss: 0.9857

## Estadística y gráficos

In [8]:
from IPython.display import display
from sklearn.metrics import classification_report, confusion_matrix

resultados_df = pd.DataFrame.from_dict(resultados, orient="index")
display(resultados_df.round(4))

fig, axes = plt.subplots(len(historiales), 2, figsize=(13, 4 * len(historiales)))
for ax, (nombre, history) in zip(axes, historiales.items()):
    epocas = range(1, len(history["loss"]) + 1)
    ax[0].plot(epocas, history["accuracy"], label="Entrenamiento")
    ax[0].plot(epocas, history["val_accuracy"], label="Validación")
    ax[0].set(title=f"{nombre}: accuracy", xlabel="Época", ylabel="Accuracy")
    ax[0].legend()
    ax[0].grid(alpha=0.25)
    ax[1].plot(epocas, history["loss"], label="Entrenamiento")
    ax[1].plot(epocas, history["val_loss"], label="Validación")
    ax[1].set(title=f"{nombre}: pérdida", xlabel="Época", ylabel="Pérdida")
    ax[1].legend()
    ax[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

ax = resultados_df[["val_accuracy", "test_accuracy"]].plot(
    kind="bar", figsize=(10, 5), ylim=(0, 1), rot=15
)
ax.set(title="Comparación de accuracy", ylabel="Accuracy", xlabel="Modelo")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

nombres_clases = [
    "avión", "automóvil", "pájaro", "gato", "ciervo",
    "perro", "rana", "caballo", "barco", "camión",
]
filas_clasificacion = []
matrices_confusion = {}
for nombre, modelo in modelos.items():
    probabilidades = modelo.predict(x_test, batch_size=BATCH_SIZE, verbose=0)
    predicciones = np.argmax(probabilidades, axis=1)
    reporte = classification_report(
        y_test,
        predicciones,
        labels=range(NUM_CLASSES),
        target_names=nombres_clases,
        output_dict=True,
        zero_division=0,
    )
    filas_clasificacion.append(
        {
            "modelo": nombre,
            "precision_macro": reporte["macro avg"]["precision"],
            "recall_macro": reporte["macro avg"]["recall"],
            "f1_macro": reporte["macro avg"]["f1-score"],
        }
    )
    matrices_confusion[nombre] = confusion_matrix(
        y_test, predicciones, labels=range(NUM_CLASSES), normalize="true"
    )

display(pd.DataFrame(filas_clasificacion).set_index("modelo").round(4))
fig, axes = plt.subplots(
    1, len(matrices_confusion), figsize=(18, 5), layout="constrained"
)
for ax, (nombre, matriz) in zip(axes, matrices_confusion.items()):
    imagen = ax.imshow(matriz, cmap="Blues", vmin=0, vmax=1)
    ax.set(title=nombre, xlabel="Predicción", ylabel="Etiqueta real")
    ax.set_xticks(range(NUM_CLASSES), nombres_clases, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES), nombres_clases)
fig.colorbar(imagen, ax=axes, label="Proporción por clase", shrink=0.8)
plt.show()

,train_accuracy,val_accuracy,test_accuracy,test_loss,duracion_min,epoca_mejor_validacion
CNN básica,0.7274,0.7434,0.7391,0.7614,2.7771,10
CNN residual,0.7326,0.7180,0.7086,0.8649,3.7673,10
MobileNetV2 (transferencia),0.7026,0.8298,0.8269,0.5076,4.6038,6


<Figure size 1300x1200 with 6 Axes>

<Figure size 1000x500 with 1 Axes>

,precision_macro,recall_macro,f1_macro
modelo,,,
CNN básica,0.7509,0.7391,0.7315
CNN residual,0.7427,0.7086,0.7064
MobileNetV2 (transferencia),0.8313,0.8269,0.8251


/tmp/ipykernel_379591/3866026183.py:68: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


<Figure size 1800x500 with 4 Axes>

# Conclusiones.

Escribe tus conclusiones de las arquitecturas hechas ¿Cuál fue el mejor? ¿Por qué? ¿Qué mejoraría? ¿Cómo lo mejoraría?

In [9]:
from IPython.display import Markdown, display

mejor_modelo = resultados_df["val_accuracy"].idxmax()
mejores_metricas = resultados_df.loc[mejor_modelo]
brecha = mejores_metricas["train_accuracy"] - mejores_metricas["val_accuracy"]

if brecha > 0.10:
    diagnostico = (
        f"La brecha entrenamiento-validación ({brecha:.1%}) sugiere sobreajuste; "
        "conviene reforzar la regularización o el aumento de datos."
    )
elif brecha > 0.03:
    diagnostico = (
        f"La brecha entrenamiento-validación es moderada ({brecha:.1%}); "
        "se debe vigilar el sobreajuste en más épocas."
    )
else:
    diagnostico = (
        f"La brecha entrenamiento-validación es pequeña ({brecha:.1%}); "
        "no se observa una señal fuerte de sobreajuste con este criterio."
    )

if "MobileNetV2" in mejor_modelo:
    recomendacion = (
        "Como MobileNetV2 se entrenó con la base congelada, un siguiente paso "
        "sería descongelar sus últimas capas y ajustar con una tasa de aprendizaje menor."
    )
else:
    recomendacion = (
        "Se puede comparar el resultado con una versión de MobileNetV2 afinada "
        "descongelando algunas capas finales de la base preentrenada."
    )

conclusion = f"### Conclusiones del experimento\n\n"
conclusion += (
    f"- El modelo con mayor accuracy de validación fue **{mejor_modelo}** "
    f"({mejores_metricas.val_accuracy:.1%}); su accuracy en prueba fue "
    f"**{mejores_metricas.test_accuracy:.1%}**.\n"
)
conclusion += f"- {diagnostico}\n"
conclusion += (
    "- La comparación incluye precisión, recall y F1 macro, además de las "
    "matrices de confusión para identificar las clases que más se confunden.\n"
)
conclusion += (
    "- La selección se basa en validación; el conjunto de prueba se reserva "
    "para estimar el desempeño final.\n"
)
conclusion += f"- {recomendacion}\n"
conclusion += (
    f"\nEstos resultados corresponden a CIFAR-10, {EPOCHS} épocas como "
    "máximo y aumento de datos en entrenamiento. Para conclusiones "
    "definitivas, considera repetir el experimento con varias semillas y "
    "comparar también el tiempo de entrenamiento."
)
display(Markdown(conclusion))

### Conclusiones del experimento

- El modelo con mayor accuracy de validación fue **MobileNetV2 (transferencia)** (83.0%); su accuracy en prueba fue **82.7%**.
- La brecha entrenamiento-validación es pequeña (-12.7%); no se observa una señal fuerte de sobreajuste con este criterio.
- La comparación incluye precisión, recall y F1 macro, además de las matrices de confusión para identificar las clases que más se confunden.
- La selección se basa en validación; el conjunto de prueba se reserva para estimar el desempeño final.
- Como MobileNetV2 se entrenó con la base congelada, un siguiente paso sería descongelar sus últimas capas y ajustar con una tasa de aprendizaje menor.

Estos resultados corresponden a CIFAR-10, 10 épocas como máximo y aumento de datos en entrenamiento. Para conclusiones definitivas, considera repetir el experimento con varias semillas y comparar también el tiempo de entrenamiento.